In [4]:
import pandas as pd
import numpy as np
import psutil
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score, classification_report, precision_recall_curve
from catboost import CatBoostClassifier, Pool
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [3]:
FILE_PATH = "../data/raw/train.csv"
HOLD_OUT_SIZE = 1_000_000

print(f"Свободно ОЗУ: {psutil.virtual_memory().available / (1024**3):.1f} ГБ")

df = pd.read_csv(FILE_PATH)
train_df = df.iloc[:-HOLD_OUT_SIZE].copy().reset_index(drop=True)
test_df = df.iloc[-HOLD_OUT_SIZE:].copy().reset_index(drop=True)
test_locked = test_df.copy()
del df

def safe_downcast(df):
    """Автоматически понижает типы данных до минимально безопасных"""
    for col in df.select_dtypes(include=['integer']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes(include=['floating']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include=['object', 'string']).columns:
        if df[col].nunique() < 50:
            df[col] = df[col].astype('category')
    return df

train_df = safe_downcast(train_df)
test_df = safe_downcast(test_df)
test_locked = safe_downcast(test_locked)

train_df = train_df.drop('id', axis=1)
test_df = test_df.drop('id', axis=1)
test_locked = test_locked.drop('id', axis=1)

print(f"Train: {len(train_df):,} | Test: {len(test_df):,} | Locked: {len(test_locked):,}")
print(f"Память Train: {train_df.memory_usage(deep=True).sum() / (1024**2):.1f} МБ")

Свободно ОЗУ: 6.5 ГБ
Train: 10,504,798 | Test: 1,000,000 | Locked: 1,000,000
Память Train: 210.4 МБ


In [4]:
def target_encode(train, test, cat_cols, target_col='Response', smoothing=10):
    """Target Encoding с Bayesian smoothing"""
    global_mean = train[target_col].mean()
    for col in cat_cols:
        means = train.groupby(col)[target_col].mean()
        counts = train.groupby(col)[target_col].count()
        smoothed = (counts * means + smoothing * global_mean) / (counts + smoothing)
        train[f'{col}_target_enc'] = train[col].map(smoothed).fillna(global_mean)
        test[f'{col}_target_enc'] = test[col].map(smoothed).fillna(global_mean)
    return train, test

def add_aggregations(train, test, group_cols=['Region_Code', 'Policy_Sales_Channel']):
    """Среднее и std Annual_Premium внутри категорий"""
    for col in group_cols:
        train[f'Prem_mean_{col}'] = train.groupby(col)['Annual_Premium'].transform('mean')
        train[f'Prem_std_{col}'] = train.groupby(col)['Annual_Premium'].transform('std').fillna(0)
        test[f'Prem_mean_{col}'] = test.groupby(col)['Annual_Premium'].transform('mean')
        test[f'Prem_std_{col}'] = test.groupby(col)['Annual_Premium'].transform('std').fillna(0)
    return train, test

def add_interactions(train, test):
    """Взаимодействия признаков"""
    train['Prem_per_Age'] = train['Annual_Premium'] / (train['Age'] + 1)
    test['Prem_per_Age'] = test['Annual_Premium'] / (test['Age'] + 1)
    train['Age_x_Vintage'] = train['Age'] * train['Vintage']
    test['Age_x_Vintage'] = test['Age'] * test['Vintage']
    return train, test

def manual_encode(train, test):
    """Ручное кодирование + удаление исходных категориальных столбцов"""
    age_map = {'< 1 Year': 0, '1-2 Year': 1, '> 2 Years': 2}
    train['Vehicle_Age_enc'] = train['Vehicle_Age'].map(age_map).astype(int)
    test['Vehicle_Age_enc'] = test['Vehicle_Age'].map(age_map).astype(int)
    train['Vehicle_Damage_enc'] = (train['Vehicle_Damage'] == 'Yes').astype(int)
    test['Vehicle_Damage_enc'] = (test['Vehicle_Damage'] == 'Yes').astype(int)
    train['Gender_enc'] = (train['Gender'] == 'Male').astype(int)
    test['Gender_enc'] = (test['Gender'] == 'Male').astype(int)
    # Удаляем исходные категориальные столбцы, чтобы не было конфликта типов
    cols_to_drop = ['Vehicle_Age', 'Vehicle_Damage', 'Gender']
    train = train.drop(columns=[c for c in cols_to_drop if c in train.columns])
    test = test.drop(columns=[c for c in cols_to_drop if c in test.columns])
    return train, test

def group_rare(df, col, threshold=0.01):
    """Группировка редких категорий в 'Other'. Приводит столбец к строке."""
    df[col] = df[col].astype(str)  # Приводим к строке, чтобы не было смешанных типов
    counts = df[col].value_counts(normalize=True)
    rare_cats = counts[counts < threshold].index.tolist()
    df[col] = df[col].apply(lambda x: 'Other' if x in rare_cats else x)
    return df

def frequency_encode(train, test, cat_cols):
    """Заменяет категорию на частоту её встречаемости"""
    for col in cat_cols:
        freq = train[col].value_counts(normalize=True)
        train[f'{col}_freq'] = train[col].map(freq)
        test[f'{col}_freq'] = test[col].map(freq).fillna(freq.mean())
    return train, test

def woe_encode(train, test, cat_cols, target_col='Response'):
    """Weight of Evidence encoding для бинарной классификации"""
    for col in cat_cols:
        # Считаем распределение классов внутри каждой категории
        cross_tab = pd.crosstab(train[col], train[target_col])
        epsilon = 1e-6 # Добавляем epsilon чтобы избежать деления на ноль
        pos_rate = (cross_tab[1] + epsilon) / (cross_tab[1].sum() + epsilon)
        neg_rate = (cross_tab[0] + epsilon) / (cross_tab[0].sum() + epsilon)
        
        # WOE = ln(neg_rate / pos_rate)
        woe = np.log(neg_rate / pos_rate)
        
        train[f'{col}_woe'] = train[col].map(woe)
        test[f'{col}_woe'] = test[col].map(woe).fillna(0)
    return train, test

In [5]:
# ВАРИАНТ 1: FULL (основной = TE + агрегации + взаимодействия)
v1_train = train_df.copy()
v1_test = test_df.copy()
v1_train, v1_test = target_encode(v1_train, v1_test, ['Region_Code', 'Policy_Sales_Channel'])
v1_train, v1_test = add_aggregations(v1_train, v1_test)
v1_train, v1_test = add_interactions(v1_train, v1_test)
v1_train, v1_test = manual_encode(v1_train, v1_test)
print(f"Вариант 1 (Target Encoding): {len([c for c in v1_train.columns if c not in ['Response']])} признаков")

# ВАРИАНТ 2: OHE (One-Hot вместо TE + агрегации + взаимодействия)
v2_train = train_df.copy()
v2_test = test_df.copy()

# 1. Сначала группируем редкие категории в категорию OTher
v2_train = group_rare(v2_train, 'Region_Code', threshold=0.01)
v2_test = group_rare(v2_test, 'Region_Code', threshold=0.01)
v2_train = group_rare(v2_train, 'Policy_Sales_Channel', threshold=0.01)
v2_test = group_rare(v2_test, 'Policy_Sales_Channel', threshold=0.01)

# 2. Потом OHE и дроп исходных категориальных столбцов
v2_train, v2_test = add_aggregations(v2_train, v2_test)
v2_train, v2_test = add_interactions(v2_train, v2_test)
ohe_cols = ['Region_Code', 'Policy_Sales_Channel']
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
ohe.fit(v2_train[ohe_cols])
v2_train_ohe = pd.DataFrame(ohe.transform(v2_train[ohe_cols]), columns=ohe.get_feature_names_out(), index=v2_train.index)
v2_test_ohe = pd.DataFrame(ohe.transform(v2_test[ohe_cols]), columns=ohe.get_feature_names_out(), index=v2_test.index)

v2_train = pd.concat([v2_train.drop(ohe_cols, axis=1), v2_train_ohe], axis=1)
v2_test = pd.concat([v2_test.drop(ohe_cols, axis=1), v2_test_ohe], axis=1)
v2_train, v2_test = manual_encode(v2_train, v2_test)
print(f"Вариант 2 (OHE): {len([c for c in v2_train.columns if c not in ['Response']])} признаков")

# ВАРИАНТ 3: Frequency Encoding (частота встречаемости категории)
v3_train = train_df.copy()
v3_test = test_df.copy()
v3_train, v3_test = frequency_encode(v3_train, v3_test, ['Region_Code', 'Policy_Sales_Channel'])
v3_train, v3_test = add_aggregations(v3_train, v3_test)
v3_train, v3_test = add_interactions(v3_train, v3_test)
v3_train, v3_test = manual_encode(v3_train, v3_test)
# Удаляем исходные категориальные столбцы
v3_train = v3_train.drop(['Region_Code', 'Policy_Sales_Channel'], axis=1)
v3_test = v3_test.drop(['Region_Code', 'Policy_Sales_Channel'], axis=1)
print(f"Вариант 3 (Frequency): {len([c for c in v3_train.columns if c not in ['Response']])} признаков")

# ВАРИАНТ 4: WOE Encoding (учитывает соотношение классов внутри категории)
v4_train = train_df.copy()
v4_test = test_df.copy()
v4_train, v4_test = woe_encode(v4_train, v4_test, ['Region_Code', 'Policy_Sales_Channel'])
v4_train, v4_test = add_aggregations(v4_train, v4_test)
v4_train, v4_test = add_interactions(v4_train, v4_test)
v4_train, v4_test = manual_encode(v4_train, v4_test)
v4_train = v4_train.drop(['Region_Code', 'Policy_Sales_Channel'], axis=1)
v4_test = v4_test.drop(['Region_Code', 'Policy_Sales_Channel'], axis=1)
print(f"Вариант 4 (WOE): {len([c for c in v4_train.columns if c not in ['Response']])} признаков")

# ВАРИАНТ 5: Бенчмарк (ручное кодирование, сырые регионы и каналы продаж)
v5_train = train_df.copy()
v5_test = test_df.copy()
v5_train, v5_test = manual_encode(v5_train, v5_test)
print(f"Вариант 5 (Бенчмарк): {len([c for c in v5_train.columns if c not in ['Response']])} признаков")

Вариант 1 (Target Encoding): 18 признаков
Вариант 2 (OHE): 45 признаков
Вариант 3 (Frequency): 16 признаков
Вариант 4 (WOE): 16 признаков
Вариант 5 (Бенчмарк): 10 признаков


In [6]:
# Применяем downcasting ко всем вариантам
v1_train = safe_downcast(v1_train); v1_test = safe_downcast(v1_test)
v2_train = safe_downcast(v2_train); v2_test = safe_downcast(v2_test)
v3_train = safe_downcast(v3_train); v3_test = safe_downcast(v3_test)
v4_train = safe_downcast(v4_train); v4_test = safe_downcast(v4_test)
v5_train = safe_downcast(v5_train); v5_test = safe_downcast(v5_test)

# Списки признаков для каждого варианта
v1_features = [c for c in v1_train.columns if c != 'Response']
v2_features = [c for c in v2_train.columns if c != 'Response']
v3_features = [c for c in v3_train.columns if c != 'Response']
v4_features = [c for c in v4_train.columns if c != 'Response']
v5_features = [c for c in v5_train.columns if c != 'Response']

# Категориальные признаки для нативного CatBoost (только для V5 Benchmark)
v5_cat_features = ['Region_Code', 'Policy_Sales_Channel']

print(f"\nРазмеры вариантов:")
print(f"V1 (Target Encoding): {len(v1_features)} признаков")
print(f"V2 (One-Hot): {len(v2_features)} признаков")
print(f"V3 (Frequency): {len(v3_features)} признаков")
print(f"V4 (WOE): {len(v4_features)} признаков")
print(f"V5 (Benchmark): {len(v5_features)} признаков (+ cat_features)")

print(f"\nПамять после downcasting:")
for name, df in [('V1', v1_train), ('V2', v2_train), ('V3', v3_train), ('V4', v4_train), ('V5', v5_train)]:
    print(f"{name}: {df.memory_usage(deep=True).sum() / (1024**2):.1f} МБ")


Размеры вариантов:
V1 (Target Encoding): 18 признаков
V2 (One-Hot): 45 признаков
V3 (Frequency): 16 признаков
V4 (WOE): 16 признаков
V5 (Benchmark): 10 признаков (+ cat_features)

Память после downcasting:
V1: 591.1 МБ
V2: 1673.0 МБ
V3: 510.9 МБ
V4: 510.9 МБ
V5: 210.4 МБ


In [8]:
# Вариант 1: Target Encoding
v1_train[v1_features + ['Response']].to_parquet('../data/processed/v1_te.parquet', index=False)
v1_test[v1_features].to_parquet('../data/processed/v1_te_test.parquet', index=False)

# Вариант 2: One-Hot Encoding
v2_train[v2_features + ['Response']].to_parquet('../data/processed/v2_ohe.parquet', index=False)
v2_test[v2_features].to_parquet('../data/processed/v2_ohe_test.parquet', index=False)

# Вариант 3: Frequency Encoding
v3_train[v3_features + ['Response']].to_parquet('../data/processed/v3_freq.parquet', index=False)
v3_test[v3_features].to_parquet('../data/processed/v3_freq_test.parquet', index=False)

# Вариант 4: WOE Encoding
v4_train[v4_features + ['Response']].to_parquet('../data/processed/v4_woe.parquet', index=False)
v4_test[v4_features].to_parquet('../data/processed/v4_woe_test.parquet', index=False)

# Вариант 5: Benchmark (сырые категории регионов и каналов продаж для CatBoost)
v5_train[v5_features + ['Response']].to_parquet('../data/processed/v5_benchmark.parquet', index=False)
v5_test[v5_features].to_parquet('../data/processed/v5_benchmark_test.parquet', index=False)

# Locked test — для финальной проверки (общий для всех вариантов)
test_locked.to_parquet('../data/processed/test_locked.parquet', index=False)

print("Сохранено в data/processed/...")

Сохранено в data/processed/...


In [9]:
DATA_DIR = Path("../data/processed")

VARIANTS = {
    "v1_te":        {"train": "v1_te.parquet",        "test": "v1_te_test.parquet",        "cat_features": []},
    "v2_ohe":       {"train": "v2_ohe.parquet",       "test": "v2_ohe_test.parquet",       "cat_features": []},
    "v3_freq":      {"train": "v3_freq.parquet",      "test": "v3_freq_test.parquet",      "cat_features": []},
    "v4_woe":       {"train": "v4_woe.parquet",       "test": "v4_woe_test.parquet",       "cat_features": []},
    "v5_benchmark": {"train": "v5_benchmark.parquet", "test": "v5_benchmark_test.parquet",
                     "cat_features": ["Region_Code", "Policy_Sales_Channel"]},
}


def load_variant(name, data_dir=DATA_DIR):
    """Возвращает (train, test, features, cat_features)."""
    cfg = VARIANTS[name]
    train = pd.read_parquet(Path(data_dir) / cfg["train"])
    test = pd.read_parquet(Path(data_dir) / cfg["test"])
    cat = cfg["cat_features"]
    if cat:  # CatBoost требует str для нативных категорий
        train[cat] = train[cat].astype(str)
        test[cat] = test[cat].astype(str)
    features = [c for c in train.columns if c != "Response"]
    return train, test, features, cat


def combine_variants(parts, data_dir=DATA_DIR):
    """Собирает новый датасет из колонок разных вариантов (строки выровнены).
    parts = [("v5_benchmark", None),                       # None = все признаки
             ("v1_te", ["Region_Code_target_enc"])]        # список = только эти колонки
    Response берётся из первого варианта.
    """
    tr_out, te_out, n = None, None, None
    for name, cols in parts:
        tr, te, feats, _ = load_variant(name, data_dir)
        if cols is None:
            cols = feats
        if tr_out is None:
            n = len(tr)
            tr_out = tr[cols + ["Response"]].copy()
            te_out = te[cols].copy()
        else:
            assert len(tr) == n, "Строки не выровнены!"
            new = [c for c in cols if c not in tr_out.columns]
            tr_out = pd.concat([tr_out, tr[new].reset_index(drop=True)], axis=1)
            te_out = pd.concat([te_out, te[new].reset_index(drop=True)], axis=1)
    return tr_out, te_out

In [10]:
"""Масштабирование числовых признаков (ТЗ п.9).
Скалеры обучаются ТОЛЬКО на train и сохраняются как артефакты —
это защищает от leakage и позволяет применять их к любым новым данным.
Для CatBoost масштабирование не нужно; для LogReg/RF/SVM — обязательно.
"""
ART_DIR = Path("artifacts/scalers")


def fit_scalers(train, features, cat_features, variant, out_dir=ART_DIR):
    """Fit Standard + MinMax на train, сохраняет артефакты."""
    num = [c for c in features if c not in cat_features]
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    for method, cls in [("standard", StandardScaler), ("minmax", MinMaxScaler)]:
        sc = cls()
        sc.fit(train[num])
        joblib.dump({"scaler": sc, "num_features": num}, out / f"{variant}_{method}.joblib")
    return num


def apply_scaler(df, variant, method="standard", art_dir=ART_DIR):
    """Применяет сохранённый скалер (train-статистики) к любому датафрейму."""
    bundle = joblib.load(Path(art_dir) / f"{variant}_{method}.joblib")
    out = df.copy()
    num = bundle["num_features"]
    out[num] = bundle["scaler"].transform(out[num])
    return out


def analyze_correlations(train, variant, target="Response",
                         sample=500_000, corr_threshold=0.9, out_dir="reports"):
    """Корреляционный анализ (ТЗ п.5): heatmap + мультиколлинеарность + связь с таргетом.
    Считается на сэмпле (корреляции стабильны уже на 500k строк)."""
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    df = train.sample(n=min(sample, len(train)), random_state=42)
    num = df.select_dtypes(include=[np.number]).columns.tolist()
    corr = df[num].corr()

    # 1. Heatmap
    size = max(8, len(num) * 0.4)
    plt.figure(figsize=(size, size))
    sns.heatmap(corr, cmap="coolwarm", center=0, square=True, linewidths=.3)
    plt.title(f"Correlation matrix — {variant}")
    plt.tight_layout()
    plt.savefig(out / f"corr_{variant}.png", dpi=150)
    plt.close()

    # 2. Пары с высокой корреляцией (мультиколлинеарность)
    cols = [c for c in num if c != target]
    pairs = []
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            r = corr.loc[a, b]
            if abs(r) >= corr_threshold:
                pairs.append({"feature_1": a, "feature_2": b, "corr": round(r, 4)})
    pairs_df = (pd.DataFrame(pairs).sort_values("corr", key=abs, ascending=False)
                if pairs else pd.DataFrame(columns=["feature_1", "feature_2", "corr"]))
    pairs_df.to_csv(out / f"multicollinearity_{variant}.csv", index=False)

    # 3. Корреляции с таргетом (слабые признаки)
    if target in corr.columns:
        tgt = corr[target].drop(target).sort_values(key=abs, ascending=False)
        tgt.to_csv(out / f"target_corr_{variant}.csv", header=["corr_with_target"])

    return {"n_high_corr_pairs": len(pairs_df), "pairs": pairs_df}

In [11]:
# Проверка матриц и скалеры для всех 5 вариантов
for name in VARIANTS:
    train, test, features, cat = load_variant(name)
    report = analyze_correlations(train, name)
    fit_scalers(train, features, cat, name)
    print(f"{name}: пар с |r|>=0.9: {report['n_high_corr_pairs']}, скалеры сохранены")
    del train, test  # экономим память (V2 весит 1.6 ГБ)

# 3. Проверка комбинирования: V5 + target_enc из V1
tr, te = combine_variants([("v5_benchmark", None), ("v1_te", None),])
print(f"Комбинированный датасет: {tr.shape}, {te.shape}")

# 4. Проверка скалера
train, test, features, cat = load_variant("v1_te")
scaled = apply_scaler(test.head(5), "v1_te", method="standard")
print(scaled[["Age", "Annual_Premium"]].head())

v1_te: пар с |r|>=0.9: 0, скалеры сохранены
v2_ohe: пар с |r|>=0.9: 1, скалеры сохранены
v3_freq: пар с |r|>=0.9: 0, скалеры сохранены
v4_woe: пар с |r|>=0.9: 0, скалеры сохранены
v5_benchmark: пар с |r|>=0.9: 0, скалеры сохранены
Комбинированный датасет: (10504798, 19), (1000000, 18)
        Age  Annual_Premium
0 -0.892583        0.069700
1 -0.959278       -0.394666
2  2.108708        0.492155
3  0.908191       -1.690456
4  2.108708        2.337775


In [45]:
# Функция для обучения и получения importance
def analyze_feature_importance(X_train, y_train, X_test, y_test, variant_name, cat_features=None):
    print(f"\n📊 {variant_name}")
    model = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, auto_class_weights='Balanced',
                               eval_metric='F1', random_seed=63, verbose=0)
    if cat_features:
        model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_test, y_test))
    else:
        model.fit(X_train, y_train, eval_set=(X_test, y_test))
    importance = model.feature_importances_
    feature_names = X_train.columns if hasattr(X_train, 'columns') else range(len(importance))
    
    feat_imp_df = pd.DataFrame({'feature': feature_names,'importance': importance}).sort_values('importance', ascending=False)
    
    print(f"Топ-10 признаков по Gain Importance:")
    for i, row in feat_imp_df.head(10).iterrows():
        print(f"  {i+1}. {row['feature']}: {row['importance']:.2f}%")
    
    return model, feat_imp_df

In [ ]:
results = {}

# V1: Target Encoding
model_v1, imp_v1 = analyze_feature_importance(v1_train[v1_features], v1_train['Response'],
    v1_test[v1_features], test_locked['Response'], "V1: Target Encoding")
results['V1'] = imp_v1

# V2: One-Hot Encoding
model_v2, imp_v2 = analyze_feature_importance(v2_train[v2_features], v2_train['Response'],
    v2_test[v2_features], test_locked['Response'], "V2: One-Hot Encoding")
results['V2'] = imp_v2

# V3: Frequency Encoding
model_v3, imp_v3 = analyze_feature_importance(v3_train[v3_features], v3_train['Response'],
    v3_test[v3_features], test_locked['Response'], "V3: Frequency Encoding")
results['V3'] = imp_v3

# V4: WOE Encoding
model_v4, imp_v4 = analyze_feature_importance(v4_train[v4_features], v4_train['Response'],
    v4_test[v4_features], test_locked['Response'],"V4: WOE Encoding")
results['V4'] = imp_v4

# V5: Benchmark (регион и канал продаж никак не кодируются)
v5_cat = ['Region_Code', 'Policy_Sales_Channel']
v5_train[v5_cat] = v5_train[v5_cat].astype(str)
v5_test[v5_cat] = v5_test[v5_cat].astype(str)
if 'Response' in test_locked.columns:
    test_locked[v5_cat] = test_locked[v5_cat].astype(str)

model_v5 = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, auto_class_weights='Balanced',
                              eval_metric='F1', cat_features=v5_cat, random_seed=63, verbose=0)
model_v5.fit(v5_train[v5_features], v5_train['Response'], 
             eval_set=(v5_test[v5_features], v5_test['Response']))

model_v5, imp_v5 = analyze_feature_importance(v5_train[v5_features], v5_train['Response'],
                                              v5_test[v5_features], test_locked['Response'],
                                              "V5: Benchmark (Native)",
                                              cat_features=v5_cat)
results['V5'] = imp_v5


📊 V1: Target Encoding
Топ-10 признаков по Gain Importance:
  4. Previously_Insured: 45.24%
  17. Vehicle_Damage_enc: 13.95%
  9. Policy_Sales_Channel_target_enc: 10.45%
  7. Vintage: 10.00%
  1. Age: 9.05%
  8. Region_Code_target_enc: 2.37%
  6. Policy_Sales_Channel: 2.02%
  11. Prem_std_Region_Code: 1.25%
  10. Prem_mean_Region_Code: 0.96%
  16. Vehicle_Age_enc: 0.92%

📊 V2: One-Hot Encoding
Топ-10 признаков по Gain Importance:
  3. Previously_Insured: 45.47%
  44. Vehicle_Damage_enc: 14.85%
  1. Age: 11.36%
  5. Vintage: 9.24%
  36. Policy_Sales_Channel_152.0: 3.20%
  9. Prem_std_Policy_Sales_Channel: 3.07%
  7. Prem_std_Region_Code: 1.74%
  40. Policy_Sales_Channel_160.0: 1.57%
  8. Prem_mean_Policy_Sales_Channel: 1.53%
  6. Prem_mean_Region_Code: 1.23%

📊 V3: Frequency Encoding
Топ-10 признаков по Gain Importance:
  3. Previously_Insured: 45.68%
  15. Vehicle_Damage_enc: 14.73%
  1. Age: 10.71%
  5. Vintage: 9.08%
  11. Prem_std_Policy_Sales_Channel: 5.37%
  7. Policy_Sales_Channe

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_recall_curve
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

print("Финальное сравнение вариантов на test_locked...")

# Конфигурация: указываем train/test наборы и особенности
variants = {'V1_TE': {'train': v1_train, 'test': v1_test, 'features': v1_features, 'cat_features': None},
            'V2_OHE': {'train': v2_train, 'test': v2_test, 'features': v2_features, 'cat_features': None},
            'V3_Freq': {'train': v3_train, 'test': v3_test, 'features': v3_features, 'cat_features': None},
            'V4_WOE': {'train': v4_train, 'test': v4_test, 'features': v4_features, 'cat_features': None},
            'V5_Native': {'train': v5_train, 'test': v5_test, 'features': v5_features, 'cat_features': ['Region_Code', 'Policy_Sales_Channel']}}

results = []
best_f1 = -1
best_variant = None
best_model = None
best_preds = None
best_probs = None
best_thresh = None

for name, cfg in variants.items():
    print(f"\n🔹 {name}:")
    X_tr_full = cfg['train'][cfg['features']].copy()
    y_tr_full = cfg['train']['Response']
    X_te = cfg['test'][cfg['features']].copy()
    y_te = test_locked['Response'].values 
    
    if cfg['cat_features']:
        for col in cfg['cat_features']:
            X_tr_full[col] = X_tr_full[col].astype(str)
            X_te[col] = X_te[col].astype(str)
    X_tr, X_val, y_tr, y_val = train_test_split(X_tr_full, y_tr_full, test_size=0.2, random_state=63, stratify=y_tr_full)
    model = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, auto_class_weights='Balanced',eval_metric='F1',
        random_seed=66, verbose=0, cat_features=cfg['cat_features'])

    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
    val_probs = model.predict_proba(X_val)[:, 1]
    prec, rec, thresholds = precision_recall_curve(y_val, val_probs)
    f1_scores = 2 * (prec * rec) / (prec + rec + 1e-8)
    opt_thresh = thresholds[np.argmax(f1_scores)]
    test_probs = model.predict_proba(X_te)[:, 1]
    test_preds = (test_probs > opt_thresh).astype(int)
    f1 = f1_score(y_te, test_preds, pos_label=1)

    results.append({'Variant': name, 'F1_test': f1, 'Threshold': opt_thresh,'Best_Iteration': model.best_iteration_})
    print(f"   📊 F1: {f1:.4f} | Threshold: {opt_thresh:.3f} | Iter: {model.best_iteration_}")

    # Фиксируем победителя
    if f1 > best_f1:
        best_f1 = f1
        best_variant = name
        best_model = model
        best_preds = test_preds
        best_probs = test_probs
        best_thresh = opt_thresh

df_res = pd.DataFrame(results).sort_values('F1_test', ascending=False).reset_index(drop=True)
print()
print("ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print(df_res.to_string(index=False))
print(f"\nПОБЕДИТЕЛЬ: {best_variant} | F1 = {best_f1:.4f} | Threshold = {best_thresh:.3f}")

🏁 Финальное сравнение вариантов на test_locked...

🔹 V1_TE:
   📊 F1: 0.4856 | Threshold: 0.698 | Iter: 967

🔹 V2_OHE:
   📊 F1: 0.4823 | Threshold: 0.693 | Iter: 995

🔹 V3_Freq:
   📊 F1: 0.4826 | Threshold: 0.698 | Iter: 993

🔹 V4_WOE:
   📊 F1: 0.4855 | Threshold: 0.696 | Iter: 994

🔹 V5_Native:
   📊 F1: 0.4866 | Threshold: 0.702 | Iter: 997

🏆 ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ (test_locked)
  Variant  F1_test  Threshold  Best_Iteration
V5_Native 0.486561   0.701918             997
    V1_TE 0.485562   0.697796             967
   V4_WOE 0.485539   0.695651             994
  V3_Freq 0.482643   0.698129             993
   V2_OHE 0.482269   0.693360             995

🥇 ПОБЕДИТЕЛЬ: V5_Native | F1 = 0.4866 | Threshold = 0.702
